# 07 · 遷移學習

從零訓練一個大模型要海量資料與算力。**遷移學習**讓你站在巨人肩膀上：拿一個在百萬張圖上預訓練好的模型，借用它學到的「視覺特徵」，只花少少資料就解決你自己的任務。

## 學習目標

- 載入預訓練的 `resnet18`
- 把它當成**凍結的特徵萃取器**
- 在少量資料上，比較「遷移學習」與「從零訓練」

## 1. 載入預訓練模型

`torchvision` 提供一堆在 ImageNet（百萬張、1000 類）上訓練好的模型。`resnet18` 是輕量又經典的選擇。

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

torch.manual_seed(0)
weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)   # 下載預訓練權重（約 45MB）
resnet.eval()
print("ResNet18 載入完成，最後一層原本是:", resnet.fc)

ResNet18 載入完成，最後一層原本是: Linear(in_features=512, out_features=1000, bias=True)


## 2. 當成「特徵萃取器」

把最後的分類層換掉，前面所有層**凍結**（不訓練）。這樣 resnet 就變成一台「把影像轉成 512 維特徵向量」的機器，我們再用這些特徵接一個簡單分類器。

我們用 **CIFAR-10**（10 類自然影像：飛機、貓、船…）。它跟 resnet 預訓練的 ImageNet 同樣是自然照片，所以特徵**轉得過去**——這正是遷移學習發威的場景。為了快，只取小子集。

In [2]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# resnet 需要較大尺寸與 ImageNet 正規化；CIFAR 本來就是 3 通道彩色
tfm = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(weights.transforms().mean, weights.transforms().std),
])
train = Subset(datasets.CIFAR10("data", train=True, download=True, transform=tfm), range(1500))
test = Subset(datasets.CIFAR10("data", train=False, download=True, transform=tfm), range(500))

feature_extractor = nn.Sequential(*list(resnet.children())[:-1])  # 去掉最後的 fc
for p in feature_extractor.parameters():
    p.requires_grad = False

def extract(dataset):
    xs, ys = [], []
    with torch.no_grad():
        for xb, yb in DataLoader(dataset, batch_size=64):
            feats = feature_extractor(xb).flatten(1)   # (batch, 512)
            xs.append(feats); ys.append(yb)
    return torch.cat(xs), torch.cat(ys)

Xtr, ytr = extract(train)
Xte, yte = extract(test)
print("萃取出的特徵維度:", Xtr.shape)

  0%|          | 0.00/170M [00:00<?, ?B/s]

  0%|          | 32.8k/170M [00:00<15:19, 185kB/s]

  0%|          | 65.5k/170M [00:00<19:16, 147kB/s]

  0%|          | 98.3k/170M [00:00<21:10, 134kB/s]

  0%|          | 229k/170M [00:00<09:56, 285kB/s] 

...

 99%|█████████▉| 169M/170M [02:58<00:00, 3.22MB/s]

100%|█████████▉| 170M/170M [02:58<00:00, 3.40MB/s]

100%|█████████▉| 170M/170M [02:58<00:00, 3.59MB/s]

100%|██████████| 170M/170M [02:58<00:00, 953kB/s] 

萃取出的特徵維度: torch.Size([1500, 512])


## 3. 遷移學習 vs 從零訓練

在這 1500 張小資料上比兩種做法：
- **遷移學習**：在凍結的 resnet 特徵上，只訓練一個線性分類器。
- **從零訓練**：一個小 CNN，直接在 1500 張上學。

In [3]:
def train_clf(model, X, y, epochs=80, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad(); crit(model(X), y).backward(); opt.step()
    return model

# (a) 遷移學習：線性分類器接在預訓練特徵上
linear = train_clf(nn.Linear(512, 10), Xtr, ytr)
with torch.no_grad():
    transfer_acc = (linear(Xte).argmax(1) == yte).float().mean().item()

# (b) 從零訓練的小 CNN（用原始影像，公平起見也只看 1500 張）
raw_tfm = transforms.Compose([transforms.Resize(64), transforms.ToTensor()])
raw_train = Subset(datasets.CIFAR10("data", train=True, download=True, transform=raw_tfm), range(1500))
raw_test = Subset(datasets.CIFAR10("data", train=False, download=True, transform=raw_tfm), range(500))
small_cnn = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 16 * 16, 10),
)
opt = torch.optim.Adam(small_cnn.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
for _ in range(12):
    for xb, yb in DataLoader(raw_train, batch_size=64, shuffle=True):
        opt.zero_grad(); crit(small_cnn(xb), yb).backward(); opt.step()
small_cnn.eval(); correct = 0
with torch.no_grad():
    for xb, yb in DataLoader(raw_test, batch_size=128):
        correct += (small_cnn(xb).argmax(1) == yb).sum().item()
scratch_acc = correct / len(raw_test)

print(f"遷移學習（凍結 resnet 特徵 + 線性分類器）：{transfer_acc:.1%}")
print(f"從零訓練的小 CNN：                       {scratch_acc:.1%}")

遷移學習（凍結 resnet 特徵 + 線性分類器）：63.6%
從零訓練的小 CNN：                       44.6%


資料很少時，遷移學習通常明顯勝出——因為 resnet 早就從百萬張圖學會「邊緣、紋理、形狀」這些通用視覺特徵，不必用你的 1500 張重新學一遍。

> 進階做法叫 **fine-tuning**：不只訓練新的分類頭，還用很小的學習率微調 resnet 後面幾層，讓特徵更貼合你的任務。

## 小結

- **遷移學習** = 借用預訓練模型學到的通用特徵。
- 凍結 backbone、只換/訓練分類頭，少量資料也能有好表現。
- 資料越少，遷移學習相對「從零訓練」的優勢越大。

## 練習

1. 把訓練資料從 600 減到 100，兩種做法的差距會拉得更大嗎？
2. 試著解凍 resnet 最後一個 block，用很小的 lr 一起微調，準確率有提升嗎？

最後一課，把所有東西串成一個**完整專案**並談部署。